<!-- nav -->
[← 8.04 · TypeScript Types](08_04_TypeScript_Types.ipynb) · [Home](../../README.md) · [8.06 · TypeScript RegExp →](08_06_TypeScript_RegExp.ipynb)

# 8.05 · TypeScript Patterns

ECMAScript takes values apart where it binds or assigns names, by destructuring:

```text
const swap = ([a, b]) => [b, a];
function label({ id, name: title = "untitled", ...rest }) { ... }
[a, b] = [b, a];
```

`[a, b]` and `{ id, name: title = "untitled", ...rest }` are patterns: they look like an array and an object, but take
one apart, binding `a`, `b`, `id`, `title` and `rest`. ECMA-262 calls them binding and assignment patterns (14.3.3,
"Destructuring Binding Patterns"; 13.15.5, "Destructuring Assignment"). `TypeScript Patterns` is their dialect, its kinds
typescript-estree's, `arraypattern`, `objectpattern`, `assignmentpattern` (a default), `restelement` and TypeScript's
`tsparameterproperty`, named and shaped as mbse-programs has them
([CAP-277: TypeScript's types and destructuring patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-277.md)).
Its `Term` lists, beside them, the TypeScript5 terms a target may be: a name, a member, an object's property, a hole.

This page builds patterns from a single name up, in the builders' fluent calls, prints each as TypeScript writes it,
finds where TypeScript5's terms hold one, shows what each binds, validates them, and reads the editions that have its
kinds. Like the rest of Part I, it starts from `dialects()`; `pattern` holds a pattern's spec in the dialect's holder,
`S.TypeScript.Patterns.Expression`, and `name` writes TypeScript5's `identifier` (tutorial 4). No reader here reads
TypeScript's text, in either language: reading TypeScript is mbse-programs' (tutorial 3).

In [1]:
from mbse.Expressions.Framework import Domains, Standards, Symbolics, Terms
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, TypeScript5
from toolkit import dialects
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = dialects()
S = store.Expressions
PATTERNS = TypeScriptPatterns.Expressions.DIALECT
ToText, binds = TypeScriptPatterns.Text.ToText, TypeScriptPatterns.Expressions.bound_names
name = lambda text: lambda e: e.identifier(lambda i: i.name(text))  # TypeScript5's identifier, `text`


def pattern(spec):
    """The pattern `spec` writes, held by its dialect's holder, written in a transaction of its own."""
    with store.transaction():
        return S.TypeScript.Patterns.Expression(lambda e: e.root(spec))


def show(held):
    """A pattern's text, as TypeScript writes it, its kind, and the names it binds."""
    term = Terms.term(held)
    print(f"{ToText(held):46}", f"{Terms.kind_of(term)[0]:17}", "binds:", ", ".join(sorted(binds(term))) or "none")

<a id="T08.05.1"></a>
## 08.05.1 · A name is the smallest pattern

The smallest target is a name: `a`, TypeScript5's `identifier`, which `TypeScript Patterns`' `Term` lists, so its holder
holds one. Where it is a parameter, it may carry an annotation, a type of `TypeScript Types` (the previous page), and
be optional; a pattern binds its names, which the dialect's `bound_names` gives:

In [2]:
for spec in (name("a"),
             lambda e: e.identifier(lambda i: i.name("a").optional(True).typeAnnotation(
                 lambda t: t.typeAnnotation(lambda k: k.tsnumberkeyword(lambda h: h))))):
    held = pattern(spec)
    show(held)
    print(" " * 46, "of", Terms.dialect_of(Terms.term(held)), "| owner:", PATTERNS.owner(Terms.term(held)).name())

a                                              identifier        binds: a
                                               of Expressions.TypeScript.Patterns.Term | owner: TypeScript5
a?: number                                     identifier        binds: a
                                               of Expressions.TypeScript.Patterns.Term | owner: TypeScript5


<a id="T08.05.2"></a>
## 08.05.2 · Arrays, objects, defaults and rests

An **array** pattern, `[a, b]`, binds a sequence's items in order, its `elements`; a hole, `[a, , c]`, is an `elision`,
which skips one. An **object** pattern, `{ id, name: title }`, binds properties by their keys, its `properties` each
TypeScript5's `property`, a key and the target it binds, shorthand where they are the same name. A **default**,
`title = "untitled"`, is an `assignmentpattern`, a target `left` and an expression `right`, used where the value is
`undefined`; a **rest**, `...rest`, a `restelement`, binds what is left. They nest; the printer writes an object
pattern at the root in parentheses, as a statement holds one:

In [3]:
title = lambda d: d.assignmentpattern(lambda a: a.left(name("title")).right(
    lambda r: r.literal(lambda l: l.value(lambda v: v.as_scalar("untitled")).quote('"'))))
rest = lambda d: d.restelement(lambda r: r.argument(name("rest")))
specs = [
    lambda p: p.arraypattern(lambda a: a.elements([name("a"), name("b")])),
    lambda p: p.arraypattern(lambda a: a.elements([name("a"), lambda e: e.elision(lambda h: h), name("c")])),
    lambda p: p.objectpattern(lambda o: o.properties([
        lambda q: q.property(lambda s: s.propertyKind("init").shorthand(True).key(name("id")).value(name("id"))),
        lambda q: q.property(lambda s: s.propertyKind("init").key(name("name")).value(name("title")))])),
    title,
    lambda p: p.objectpattern(lambda o: o.properties([
        lambda q: q.property(lambda s: s.propertyKind("init").shorthand(True).key(name("id")).value(name("id"))),
        lambda q: q.property(lambda s: s.propertyKind("init").key(name("name")).value(title)),
        rest])),
    lambda p: p.arraypattern(lambda a: a.elements([
        lambda e: e.arraypattern(lambda b: b.elements([name("x"), name("y")])), rest])),
]
for spec in specs:
    show(pattern(spec))

[a, b]                                         arraypattern      binds: a, b
[a,, c]                                        arraypattern      binds: a, c
({ id, name: title })                          objectpattern     binds: id, title
title = "untitled"                             assignmentpattern binds: title
({ id, name: title = "untitled", ...rest })    objectpattern     binds: id, rest, title
[[x, y], ...rest]                              arraypattern      binds: rest, x, y


<a id="T08.05.3"></a>
## 08.05.3 · Where TypeScript5 holds one, and what it binds there

A pattern binds names through what holds it: none of its kinds is a binder of its own (tutorial 7). A function's and
an arrow's parameters are patterns, which the function binds within its body, so `([a, b]) => [b, a]` needs nothing
from its scope. An assignment's target, `[a, b] = [b, a]`, is a pattern too, but it binds nothing: it assigns names
its scope already has, references, so its names are free. TypeScript5's `Term` lists the array, object and default
patterns, which may stand where an expression does in an assignment's target; and TypeScript5 reads `TypeScript
Patterns`' terms, as each dialect of the family reads the others' (`reads()`). The properties that hold one are read
from the store:

In [4]:
swap = lambda b: b.arrayexpression(lambda a: a.elements([  # [b, a]: the framework's list, of the framework's terms
    lambda t: t.typescript5(name("b")), lambda t: t.typescript5(name("a"))]))
pair = lambda p: p.arraypattern(lambda a: a.elements([name("a"), name("b")]))  # [a, b]
with store.transaction():
    arrow = S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.arrowfunctionexpression(lambda f: f
        .params([pair]).body(swap))))
    assign = S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.assignmentexpression(lambda a: a
        .left(pair).operator("=").right(swap))))
for host in (arrow, assign):
    print(f"{TypeScript5.Text.ToText(host):20}", f"{Terms.kind_of(Terms.term(host))[0]:23}", "free:",
          ", ".join(sorted(Symbolics.free(host, TypeScript5.Expressions.DIALECT))) or "none")
patterns = store.registered(PATTERNS.union)
host = store.registered(TypeScript5.Expressions.DIALECT.union).branches
print("held by:", ", ".join(f"{b.name}.{n}" for b in host if b.name not in TypeScriptPatterns.Expressions.TAGS
                            for n, p in b.schema.properties.items() if patterns in (p.schema, getattr(p.schema, "element", None))))
print("listed by TypeScript5:", ", ".join(b.name for b in host if b.name in TypeScriptPatterns.Expressions.TAGS))
print("TypeScript5 reads", ", ".join(d.name() for d in TypeScript5.Expressions.DIALECT.reads()))

([a, b]) => [b, a]   arrowfunctionexpression free: none
[a, b] = [b, a]      assignmentexpression    free: a, b
held by: functionexpression.params, arrowfunctionexpression.params, tsemptybodyfunctionexpression.params, assignmentexpression.left
listed by TypeScript5: arraypattern, objectpattern, assignmentpattern
TypeScript5 reads TypeScript Types, TypeScript Patterns, TypeScript RegExp


A pattern alone, in its holder, is held by nothing that binds: `bound_names` says which of its names a function would
bind, but alone every name it holds is free, those as much as a default's and a member's object, which no holder
binds.

In [5]:
every = pattern(lambda p: p.objectpattern(lambda o: o.properties([  # { k: [a, , ...r] = c, m: x.y, ...z }
    lambda q: q.property(lambda s: s.propertyKind("init").key(name("k")).value(lambda v: v.assignmentpattern(lambda a: a
        .left(lambda t: t.arraypattern(lambda r: r.elements([name("a"), lambda e: e.elision(lambda h: h),
                                                             lambda e: e.restelement(lambda s: s.argument(name("r")))])))
        .right(name("c"))))),
    lambda q: q.property(lambda s: s.propertyKind("init").key(name("m")).value(lambda v: v.memberexpression(
        lambda m: m.object(name("x")).property(name("y"))))),
    lambda q: q.restelement(lambda r: r.argument(name("z")))])))
show(every)
print(" " * 46, "free:", ", ".join(sorted(Symbolics.free(every, PATTERNS))))

({ k: [a,, ...r] = c, m: x.y, ...z })          objectpattern     binds: a, r, z
                                               free: a, c, r, x, z


<a id="T08.05.4"></a>
## 08.05.4 · Validated

Validation checks each kind as typescript-estree has it, in its own words: a default needs its target and its value,
a rest its target, a name must be one TypeScript can spell, an annotation's type is checked as `TypeScript Types`
checks it, and invalid syntax where a target stands, an `error` term, is reported. A refused pattern is kept, and
printed as written
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [6]:
refused = [
    lambda p: p.arraypattern(lambda a: a.elements([name("a b")])),
    lambda p: p.arraypattern(lambda a: a.elements([name("a"), lambda e: e.error(lambda r: r.text("1 +"))])),
    lambda p: p.objectpattern(lambda o: o.typeAnnotation(lambda a: a.typeAnnotation(lambda t: t.tsuniontype(
        lambda u: u.types([lambda k: k.tsanykeyword(lambda h: h)]))))),
]
for spec in refused:
    held = pattern(spec)
    print(f"{ToText(held):13}", "; ".join(PATTERNS.validate(held, {"a"})))
assert PATTERNS.validate(pattern(refused[0]), {"a"}) == ["elements 0: an identifier's name cannot be 'a b'"]

[a b]         elements 0: an identifier's name cannot be 'a b'
[a, 1 +]      elements 1: invalid syntax '1 +'
({}: any)     typeAnnotation typeAnnotation: a tsuniontype needs two types


<a id="T08.05.5"></a>
## 08.05.5 · Inferred as any, and not simplified

A pattern is no value: a function call or an assignment matches it. Inference gives it `any`, and TypeScript5's rules fold none of `TypeScript Patterns`' kinds: each stays in the residual, named by the dialect; a name the pattern holds is TypeScript5's
kind, which TypeScript5 resolves in a scope: unbound, it stays:

In [7]:
for held in (pattern(pair), every, pattern(name("a"))):
    kind = Terms.kind_of(Terms.term(held))[0]
    try:
        TypeScript5.Simplifiers.OfAny(held).evaluate()
        raise AssertionError("evaluated")
    except NotEvaluated as error:
        print(f"{ToText(held):46}", "inferred:", PATTERNS.infer(held, dict.fromkeys(Symbolics.free(held, PATTERNS), Domains.Anything)).name(),
              "|", error)

[a, b]                                         inferred: any | the root is not a literal: TypeScript Patterns arraypattern is not simplified
({ k: [a,, ...r] = c, m: x.y, ...z })          inferred: any | the root is not a literal: TypeScript Patterns objectpattern is not simplified
a                                              inferred: any | the root is not a literal: a is not bound


<a id="T08.05.6"></a>
## 08.05.6 · Its editions as data

`TypeScript Patterns` covers TypeScript 5.0 to 5.9, each found by its published name, and each with all five of its
own kinds ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md));
the TypeScript5 terms its `Term` lists are recorded there:

In [8]:
dialect = store.Instances.Expressions.TypeScript.Patterns
editions = Standards.standards(dialect)
print(len(editions), "editions:", list(editions)[0], "to", list(editions)[-1])
print("TypeScript 5.0:", ", ".join(kind.name.rsplit(".", 1)[1] for kind in Standards.kinds(Standards.find(dialect, "TypeScript 5.0"))))

10 editions: TypeScript 5.0 to TypeScript 5.9
TypeScript 5.0: ArrayPattern, ObjectPattern, AssignmentPattern, RestElement, TSParameterProperty


## What you learned

- **A destructuring target is a term of its own dialect**, `TypeScript Patterns`: array, object, default, rest and
  parameter-property patterns, beside TypeScript5's names, members and properties that a target may be.
- **It binds through what holds it**: a function binds its parameters' names within it; an assignment's target assigns
  names its scope has, so they stay free.
- **TypeScript5 holds a pattern** in a function's parameters and an assignment's target, and lists the array, object
  and default patterns in its own `Term`.
- **Validation speaks typescript-estree's rules**; inference gives `any`, and simplification leaves each kind in the residual, named.
- **Its editions are data**: TypeScript 5.0 to 5.9.

Next, `TypeScript RegExp`: ECMAScript's regular expressions.

<!-- nav -->
[← 8.04 · TypeScript Types](08_04_TypeScript_Types.ipynb) · [Home](../../README.md) · [8.06 · TypeScript RegExp →](08_06_TypeScript_RegExp.ipynb)